# 01 — Create datasets: **one engine (Spark), one format at a time**

**Purpose:** Generate the synthetic multimodal dataset and land it in **one** of three storage
formats, all on **pure Spark** — no Ray anywhere in the data-engineering path. This is the
*format-isolation* notebook for the blog: generation, write, and backfill are held constant on
Spark so the **only** variable is the terminal storage format.

Pick the format with the `format` widget:

| `format` | Image storage | Output |
|---|---|---|
| `delta_pathref` | JPEG **files** in a Volume + an `image_path` column | table `synthetic_delta_{size}` |
| `delta_inline` | JPEG **bytes inline** in a `binary` column | table `synthetic_delta_inline_{size}` |
| `lance` | JPEG **bytes inline**, blob-isolated Lance fragments | dataset `synthetic_lance_{size}` |

The output names match exactly what `03_training_benchmark` reads and `04_compile_results`
compiles, so those notebooks are unchanged. Generation is the same deterministic `(SEED, id)`
logic for every format, wrapped in a `mapInArrow` UDF (the Arrow-batched Spark analog of a
vectorized map), so the bytes are byte-identical across the three routes.

Each run does **one** format; `02_run_benchmarks` submits the (format × size) grid as parallel
jobs, each on its own fresh cluster.

---

> ### ⚠️ Cluster requirements — read before running
>
> One Single-User **Unity-Catalog** cluster runs all three formats (the Delta paths need UC;
> the Lance connector runs fine on UC Single-User compute — the Lance dataset lands in a Volume,
> not a UC table, matching the repo's "Lance lives in Volumes" story).
>
> - **Runtime:** **DBR 16.4 LTS** (Spark 3.5 / Scala 2.12). The Lance connector bundle is built
>   for Spark 3.5 / Scala 2.12 — it will **not** load on DBR 17.x (Spark 4 / Scala 2.13). Delta
>   runs fine on 16.4, so one runtime serves all three formats.
> - **Access mode:** Single User (Dedicated). Unity Catalog enabled.
> - **Library (needed only for `format=lance`):** install
>   `org.lance:lance-spark-bundle-3.5_2.12` (Maven Central) as a cluster library, and add it to
>   the UC **allowlist** (custom Spark data source ⇒ needs `ANY FILE`).
> - **Spark config (needed only for `format=lance`):**
>   ```
>   spark.sql.catalog.lance            org.lance.spark.LanceNamespaceSparkCatalog
>   spark.sql.catalog.lance.impl       dir
>   spark.sql.catalog.lance.root       /Volumes/<catalog>/<schema>/<volume>
>   ```
>   The `dir` namespace + `root` pointed at the Volume is what lands the dataset at
>   `{volume}/synthetic_lance_{size}` — exactly where `03` reads it via `read_lance`.
> - **Backfill caveat (`format=lance`):** `ALTER TABLE … ADD COLUMNS … FROM` needs the Lance
>   Spark **SQL extension**, which may not load on Databricks. If it fails, the cell falls back
>   to a driver-side pylance `merge_columns` (marked clearly — not the pure-Spark path).
>
> These are documented, not tested here — verify on your cluster. The one integration point to
> confirm first: that the Spark-written Lance dataset lands at `{volume}/synthetic_lance_{size}`
> and reads back through `03`'s `read_lance` with matching throughput (fragment layout can
> differ from a Ray `write_fragments` write).

In [ ]:
# pylance is used only for driver-side Lance verification (fragment count, round-trip) and the
# backfill fallback. The Spark<->Lance write path itself comes from the lance-spark-bundle JAR
# installed as a CLUSTER LIBRARY (see the constraints cell) — NOT from pip.
%pip install -qU pylance numpy pandas "databricks-sdk>=0.49.0"
dbutils.library.restartPython()

In [ ]:
# ── Widgets ─────────────────────────────────────────────
dbutils.widgets.dropdown("size", "10k", ["10k", "100k", "1m", "10m"], "Dataset size")
dbutils.widgets.text("catalog", "main", "UC catalog")
dbutils.widgets.text("schema", "ml_benchmark", "UC schema")
dbutils.widgets.text("volume", "lance_benchmark", "UC volume")
dbutils.widgets.text("seed", "42", "RNG seed")
dbutils.widgets.text("embedding_dim", "512", "Embedding dim")
# One format per run — the single variable this notebook isolates.
dbutils.widgets.dropdown("format", "lance", ["delta_pathref", "delta_inline", "lance"], "Storage format")
dbutils.widgets.text("lance_namespace", "lance", "Lance Spark catalog name (matches spark.sql.catalog.<name>)")

size          = dbutils.widgets.get("size")
catalog       = dbutils.widgets.get("catalog")
schema        = dbutils.widgets.get("schema")
volume        = dbutils.widgets.get("volume")
SEED          = int(dbutils.widgets.get("seed"))
EMBEDDING_DIM = int(dbutils.widgets.get("embedding_dim"))
FORMAT        = dbutils.widgets.get("format")
LANCE_CATALOG = dbutils.widgets.get("lance_namespace")

SIZE_MAP = {"10k": 10_000, "100k": 100_000, "1m": 1_000_000, "10m": 10_000_000}
N_ROWS   = SIZE_MAP[size]

# Fixed category set — MUST match 03_training_benchmark.
CATEGORIES = ["cat", "dog", "car", "tree", "house", "flower", "boat", "bird"]

base_vol      = f"/Volumes/{catalog}/{schema}/{volume}"
images_dir    = f"{base_vol}/synthetic_images_{size}"                 # JPEG files (delta_pathref only)
delta_table   = f"{catalog}.{schema}.synthetic_delta_{size}"          # path-ref metadata table
inline_table  = f"{catalog}.{schema}.synthetic_delta_inline_{size}"   # inline-bytes table
lance_subdir  = f"synthetic_lance_{size}"                             # 03 reads read_lance(base_vol/lance_subdir)
lance_path    = f"{base_vol}/{lance_subdir}"                          # physical Lance dataset (see constraints)
lance_full    = f"{LANCE_CATALOG}.default.{lance_subdir}"             # 3-level name via the dir namespace
artifacts_dir = f"{base_vol}/artifacts"

# Artifact filename per format — kept identical to what 04_compile_results loads.
ARTIFACT_NAME = {"delta_pathref": f"delta_{size}.json",
                 "delta_inline":  f"delta_inline_{size}.json",
                 "lance":         f"lance_{size}.json"}[FORMAT]
PATH_LABEL    = {"delta_pathref": "delta_pathref",
                 "delta_inline":  "delta_inline",
                 "lance":         "lance_native"}[FORMAT]

print(f"Size tier   : {size} ({N_ROWS:,} rows)")
print(f"Format      : {FORMAT}")
print(f"Artifact    : {artifacts_dir}/{ARTIFACT_NAME}")
if FORMAT == "delta_pathref":
    print(f"Output      : table {delta_table}  +  JPEG files under {images_dir}")
elif FORMAT == "delta_inline":
    print(f"Output      : table {inline_table}")
else:
    print(f"Output      : Lance dataset {lance_path}  (via {lance_full})")

In [ ]:
# Sanity-check the Lance Spark catalog is configured — only when writing Lance. If this raises,
# the lance-spark-bundle JAR isn't installed or spark.sql.catalog.<name> is missing (constraints cell).
if FORMAT == "lance":
    cat_impl = spark.conf.get(f"spark.sql.catalog.{LANCE_CATALOG}", None)
    assert cat_impl and "Lance" in cat_impl, (
        f"Lance Spark catalog '{LANCE_CATALOG}' not configured. Install lance-spark-bundle-3.5_2.12 "
        f"and set spark.sql.catalog.{LANCE_CATALOG}=org.lance.spark.LanceNamespaceSparkCatalog "
        f"(+ .impl=dir, .root={base_vol}) in Spark config.")
    root = spark.conf.get(f"spark.sql.catalog.{LANCE_CATALOG}.root", None)
    print(f"Lance catalog impl: {cat_impl}")
    print(f"Lance namespace root: {root}  (expect {base_vol} so the dataset lands at {lance_path})")
else:
    print(f"[{FORMAT}] Lance catalog check skipped.")

In [ ]:
import os
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import catalog as sdk_catalog

w = WorkspaceClient()
try:
    w.volumes.read(f"{catalog}.{schema}.{volume}")
except Exception:
    w.volumes.create(catalog_name=catalog, schema_name=schema, name=volume,
                     volume_type=sdk_catalog.VolumeType.MANAGED)
    print(f"Created volume {catalog}.{schema}.{volume}")
os.makedirs(artifacts_dir, exist_ok=True)
if FORMAT == "delta_pathref":
    os.makedirs(images_dir, exist_ok=True)


def dir_file_sizes(path):
    """path -> {file: size}. Lets us measure exactly the files a step ADDS (backfill),
    instead of differencing whole-directory totals (which can go negative when Lance
    version cleanup drops superseded files)."""
    sizes = {}
    for root, _, files in os.walk(path):
        for f in files:
            fp = os.path.join(root, f)
            try:
                sizes[fp] = os.path.getsize(fp)
            except OSError:
                pass
    return sizes


def dir_stats(path):
    s = dir_file_sizes(path)
    return sum(s.values()), len(s)

## Generate synthetic data (once, on Spark)

Identical `(SEED, id)` generation to the rest of the benchmark — reproduced verbatim so the bytes
match — but driven entirely by Spark. `spark.range(N)` produces the id column; `mapInArrow` fans
generation across executors as an **Arrow-batched** Python UDF (vectorized, per-batch Python — the
fair analog of a `map_batches`). The image is conditioned on category (hue) so `03`'s classifier is
learnable; noise keeps the JPEG ~30–300KB.

Generation is materialized behind a `cache()` + `count()` barrier so the **write timing below
excludes generation** — the write timer measures only the format-specific commit.

In [ ]:
import numpy as np

# ── Verbatim generation logic so bytes are byte-identical across all formats (same seed -> same JPEG). ──
def _make_image(rng, category_idx, n_categories):
    """Procedural RGB image conditioned on category, JPEG-encoded to ~30-300KB."""
    import io
    from PIL import Image

    side = int(rng.integers(256, 512))
    base = np.zeros((side, side, 3), dtype=np.float32)
    hue = category_idx / n_categories
    base[..., 0] = 255 * hue
    base[..., 1] = 255 * (1 - hue)
    base[..., 2] = 128
    noise = rng.integers(0, 60, size=(side, side, 3))
    arr = np.clip(base + noise, 0, 255).astype(np.uint8)

    buf = io.BytesIO()
    Image.fromarray(arr).save(buf, format="JPEG", quality=90)
    return buf.getvalue()


def _generate_rows(ids, seed, categories, embedding_dim):
    """Per-batch generation returning plain Python lists for Arrow assembly."""
    n_cat = len(categories)
    images, captions, embeddings, cats, brightness, quality = [], [], [], [], [], []
    for _id in ids:
        rng = np.random.default_rng([seed, int(_id)])
        cat_idx = int(rng.integers(0, n_cat))
        images.append(_make_image(rng, cat_idx, n_cat))
        captions.append(f"a photo of a {categories[cat_idx]} " + "x" * int(rng.integers(0, 40)))
        embeddings.append(rng.standard_normal(embedding_dim).astype(np.float32).tolist())
        cats.append(categories[cat_idx])
        brightness.append(float(rng.random()))
        quality.append(int(rng.integers(1, 6)))
    return images, captions, embeddings, cats, brightness, quality

In [ ]:
import pyarrow as pa
from pyspark.sql.types import (
    StructType, StructField, LongType, BinaryType, StringType,
    ArrayType, FloatType, IntegerType,
)

# Generated DataFrame schema. Arrow<->Spark: BinaryType<->binary, ArrayType(FloatType)<->list<float32>.
GEN_SCHEMA = StructType([
    StructField("id",         LongType(),   False),
    StructField("image",      BinaryType(), False),
    StructField("caption",    StringType(), False),
    StructField("embedding",  ArrayType(FloatType()), False),
    StructField("category",   StringType(), False),
    StructField("brightness", FloatType(),  False),
    StructField("quality",    IntegerType(), False),
])

_SEED, _CATS, _DIM = SEED, CATEGORIES, EMBEDDING_DIM

def generate_arrow(batch_iter):
    """mapInArrow UDF: iterator of pa.RecordBatch (with an `id` column) -> generated batches."""
    for rb in batch_iter:
        ids = rb.column("id").to_pylist()
        images, captions, embeddings, cats, brightness, quality = _generate_rows(
            ids, _SEED, _CATS, _DIM)
        yield pa.record_batch({
            "id":         pa.array(ids, type=pa.int64()),
            "image":      pa.array(images, type=pa.binary()),
            "caption":    pa.array(captions, type=pa.string()),
            "embedding":  pa.array(embeddings, type=pa.list_(pa.float32())),
            "category":   pa.array(cats, type=pa.string()),
            "brightness": pa.array(brightness, type=pa.float32()),
            "quality":    pa.array(quality, type=pa.int32()),
        })

# Partition count controls generation parallelism and downstream fragment/file count
# (~5k rows/partition, floor 64) — a comparable layout across formats.
n_parts = max(64, N_ROWS // 5_000)
ids_df  = spark.range(0, N_ROWS, numPartitions=n_parts)          # column: id
gen_df  = ids_df.mapInArrow(generate_arrow, schema=GEN_SCHEMA)

# Materialize generation BEHIND A BARRIER so the write timing excludes generation.
gen_df = gen_df.cache()
row_count = gen_df.count()
raw_image_bytes = gen_df.selectExpr("sum(length(image)) AS b").collect()[0]["b"]
print(f"Generated {row_count:,} rows (cached) | raw image bytes: {raw_image_bytes / 1e9:.3f} GB")

## Write — the one format this run isolates

All three routes start from the **same cached `gen_df`** and diverge only here:

- **`delta_pathref`** — a `mapInArrow` writes each JPEG to the Volume and returns the row with an
  `image_path` (no bytes); that metadata DataFrame is written to a Delta table. Cost = the file
  PUT storm + the small metadata table.
- **`delta_inline`** — `gen_df` written straight to a Delta table with the JPEG **bytes inline** in
  a `binary` column. Every image byte funnels through the Spark write path.
- **`lance`** — `gen_df` written through the `lance` catalog with **blob encoding**
  (`image.lance.encoding=blob` + `file_format_version=2.2`) so the image bytes land in Lance's
  isolated blob layout — the apples-to-apples counterpart to the inline Delta bytes.

In [ ]:
import time

n_output_files = None
on_disk_bytes  = None

if FORMAT == "delta_pathref":
    # Write JPEG files via an Arrow UDF, returning metadata rows with image_path (no bytes).
    _images_dir = images_dir
    META_SCHEMA = StructType([
        StructField("id",         LongType(),   False),
        StructField("image_path", StringType(), False),
        StructField("caption",    StringType(), False),
        StructField("embedding",  ArrayType(FloatType()), False),
        StructField("category",   StringType(), False),
        StructField("brightness", FloatType(),  False),
        StructField("quality",    IntegerType(), False),
    ])

    def write_files_arrow(batch_iter):
        import os
        for rb in batch_iter:
            ids   = rb.column("id").to_pylist()
            imgs  = rb.column("image").to_pylist()
            paths = []
            for _id, jpeg in zip(ids, imgs):
                p = os.path.join(_images_dir, f"{int(_id):012d}.jpg")
                with open(p, "wb") as f:
                    f.write(jpeg)
                paths.append(p)
            yield pa.record_batch({
                "id":         pa.array(ids, type=pa.int64()),
                "image_path": pa.array(paths, type=pa.string()),
                "caption":    rb.column("caption"),
                "embedding":  rb.column("embedding"),
                "category":   rb.column("category"),
                "brightness": rb.column("brightness"),
                "quality":    rb.column("quality"),
            })

    spark.sql(f"DROP TABLE IF EXISTS {delta_table}")
    t0 = time.time()
    meta_df = gen_df.mapInArrow(write_files_arrow, schema=META_SCHEMA)
    # One action: writes the JPEG files AND the metadata table (files land as a side effect).
    meta_df.write.mode("overwrite").saveAsTable(delta_table)
    write_s = time.time() - t0

    img_bytes, img_files = dir_stats(images_dir)
    meta_bytes = spark.sql(f"DESCRIBE DETAIL {delta_table}").collect()[0]["sizeInBytes"] or 0
    on_disk_bytes  = int(img_bytes + meta_bytes)
    n_output_files = int(img_files + 1)                       # JPEGs + the metadata Parquet
    tgt_table      = delta_table
    print(f"delta_pathref : {write_s:6.2f}s | {img_files:,} JPEG files + 1 metadata table | "
          f"{on_disk_bytes / 1e9:.3f} GB")

elif FORMAT == "delta_inline":
    spark.sql(f"DROP TABLE IF EXISTS {inline_table}")
    t0 = time.time()
    gen_df.write.mode("overwrite").saveAsTable(inline_table)  # JPEG bytes inline in a binary column
    write_s = time.time() - t0

    _d = spark.sql(f"DESCRIBE DETAIL {inline_table}").collect()[0]
    on_disk_bytes  = int(_d["sizeInBytes"] or 0)
    n_output_files = int(_d["numFiles"] or 0)                 # Parquet files only (bytes are inline)
    tgt_table      = inline_table
    print(f"delta_inline  : {write_s:6.2f}s | bytes inline across {n_output_files} Parquet files | "
          f"{on_disk_bytes / 1e9:.3f} GB")

else:  # lance
    spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {LANCE_CATALOG}.default")
    spark.sql(f"DROP TABLE IF EXISTS {lance_full}")
    t0 = time.time()
    (gen_df.writeTo(lance_full)
        .tableProperty("image.lance.encoding", "blob")       # blob-isolated layout (not plain BINARY)
        .tableProperty("file_format_version", "2.2")
        .create())
    write_s = time.time() - t0

    # Fragment count + on-disk bytes via pylance at the expected FUSE path. If the connector
    # lands files elsewhere under the namespace root, this warns (see constraints).
    import lance
    n_frag = None
    try:
        lds = lance.dataset(lance_path)
        n_frag = len(lds.get_fragments())
        on_disk_bytes, _ = dir_stats(lance_path)
    except Exception as e:
        print(f"[warn] pylance open at {lance_path} failed ({e}); "
              f"check spark.sql.catalog.{LANCE_CATALOG}.root — dataset may be elsewhere.")
        on_disk_bytes, _ = dir_stats(lance_path)
    n_output_files = int(n_frag) if n_frag is not None else None
    tgt_table      = lance_full
    print(f"lance         : {write_s:6.2f}s | "
          f"{n_frag if n_frag is not None else '?'} fragments | "
          f"{(on_disk_bytes or 0) / 1e9:.3f} GB "
          f"(vs ~{N_ROWS:,} JPEG PUTs on the path-ref route)")

## Verify — deterministic round-trip

Regenerate the probe rows from the same `(SEED, id)` and confirm the stored bytes match
byte-for-byte — proving the Spark write is lossless and identical across formats. The read stays
pure Spark (`spark.read.table`) for Delta; Lance reads back through the catalog too.

In [ ]:
probe_ids = [0, N_ROWS // 2, N_ROWS - 1]

def _fetch_stored(pids):
    """Return {id: image_bytes} for probe ids, per format."""
    if FORMAT == "delta_pathref":
        rows = (spark.read.table(delta_table)
                .where(f"id IN ({','.join(map(str, pids))})")
                .select("id", "image_path").collect())
        out = {}
        for r in rows:
            with open(r["image_path"], "rb") as f:
                out[r["id"]] = f.read()
        return out
    tbl = inline_table if FORMAT == "delta_inline" else lance_full
    rows = (spark.read.table(tbl)
            .where(f"id IN ({','.join(map(str, pids))})")
            .select("id", "image").collect())
    return {r["id"]: bytes(r["image"]) for r in rows}

def _regen_image(pid):
    rng = np.random.default_rng([SEED, int(pid)])
    cat_idx = int(rng.integers(0, len(CATEGORIES)))
    return _make_image(rng, cat_idx, len(CATEGORIES))

stored = _fetch_stored(probe_ids)
roundtrip_ok = True
print("Round-trip (regenerated bytes == stored):")
for pid in probe_ids:
    ok = _regen_image(pid) == stored.get(pid)
    roundtrip_ok = roundtrip_ok and ok
    kb = len(stored.get(pid, b"")) / 1024
    print(f"  id={pid:>12,}: {'OK' if ok else 'MISMATCH':>8}  ({kb:.0f} KB)")
print(f"\nround-trip all OK: {roundtrip_ok}")

## ETL backfill — add a derived column (`embedding_norm`)

Add the L2 norm of the embedding — a stand-in for any derived feature — to the written dataset.
This is where the format difference is structural, all on the **same Spark engine**:

- **Delta (both)** — `ALTER TABLE ADD COLUMN` + `UPDATE`. The `UPDATE` rewrites whole Parquet row
  groups; for **inline** Delta that drags every image byte along (measured via the `UPDATE`
  operation's own `numAddedBytes`, the true rewrite cost — not the post-`UPDATE` table size,
  which double-counts the retained pre-`VACUUM` version).
- **Lance** — `ALTER TABLE … ADD COLUMNS … FROM <view>` writes **only the new column file** per
  fragment; data files untouched. Needs the Lance SQL extension (see caveat); falls back to
  driver-side pylance `merge_columns` if unavailable.

In [ ]:
backfill_via = None

if FORMAT in ("delta_pathref", "delta_inline"):
    tbl = delta_table if FORMAT == "delta_pathref" else inline_table
    _existing = {f.name for f in spark.table(tbl).schema.fields}
    t0 = time.time()
    if "embedding_norm" not in _existing:
        spark.sql(f"ALTER TABLE {tbl} ADD COLUMN embedding_norm FLOAT")
    spark.sql(f"""
        UPDATE {tbl}
        SET embedding_norm = SQRT(AGGREGATE(TRANSFORM(embedding, x -> x * x),
                                            CAST(0.0 AS DOUBLE), (acc, v) -> acc + v))
    """)
    backfill_s = time.time() - t0

    # Real bytes/files rewritten — from the UPDATE's own operationMetrics, not table size.
    _hist = spark.sql(f"DESCRIBE HISTORY {tbl}").collect()
    _upd  = next(r for r in _hist if r["operation"] == "UPDATE")
    _m    = _upd["operationMetrics"] or {}
    etl_bytes_written = int(_m.get("numAddedBytes") or _m.get("rewrittenBytes") or 0)
    etl_files         = int(_m.get("numAddedFiles") or 0)
    backfill_via = "spark_update"
    print(f"{FORMAT} backfill: {backfill_s:6.2f}s | rewrote {etl_bytes_written / 1e6:,.1f} MB "
          f"across {etl_files} files")

else:  # lance
    import lance
    before = dir_file_sizes(lance_path)
    t0 = time.time()
    try:
        # Pure-Spark path: Lance SQL extension ADD COLUMNS FROM (no row-group rewrite).
        spark.sql(f"""
            CREATE OR REPLACE TEMPORARY VIEW _norm_src AS
            SELECT _rowaddr, _fragid,
                   CAST(sqrt(aggregate(transform(embedding, x -> x * x),
                                       CAST(0.0 AS DOUBLE), (acc, v) -> acc + v)) AS FLOAT) AS embedding_norm
            FROM {lance_full}
        """)
        spark.sql(f"ALTER TABLE {lance_full} ADD COLUMNS embedding_norm FROM _norm_src")
        backfill_via = "spark_sql_extension"
    except Exception as e:
        print(f"[warn] Spark ADD COLUMNS failed ({e}); falling back to pylance merge_columns (NOT pure Spark).")
        def compute_norm(record_batch):
            embs = np.stack(record_batch.column("embedding").to_pylist()).astype("float32")
            norms = np.linalg.norm(embs, axis=1).astype("float32")
            return pa.record_batch({"embedding_norm": pa.array(norms)})
        lds_rw = lance.dataset(lance_path)
        if "embedding_norm" in lds_rw.schema.names:
            lds_rw.drop_columns(["embedding_norm"]); lds_rw = lance.dataset(lance_path)
        lds_rw.add_columns(compute_norm, columns=["embedding"])
        backfill_via = "pylance_fallback"
    backfill_s = time.time() - t0

    after = dir_file_sizes(lance_path)
    etl_bytes_written = sum(sz for fp, sz in after.items() if fp not in before)  # new files only
    etl_files = len([fp for fp in after if fp not in before])
    print(f"lance backfill : {backfill_s:6.2f}s | +{etl_bytes_written / 1e6:,.1f} MB new-column bytes "
          f"| via {backfill_via}")

In [ ]:
import json

# ── common block ── identical key names across all formats so 04_compile_results stacks them
# into one table with no per-format mapping. Format-specific detail stays in `raw`.
# Generation ran behind the cache() barrier, so write_total_s == target_write_s (no separate
# file-landing step timed apart from the write).
common = {
    "path_label":        PATH_LABEL,
    "write_total_s":     round(write_s, 3),
    "target_write_s":    round(write_s, 3),
    "n_output_files":    n_output_files,
    "on_disk_bytes":     int(on_disk_bytes) if on_disk_bytes is not None else None,
    "etl_backfill_s":    round(backfill_s, 3),
    "etl_bytes_written": int(etl_bytes_written),
    "roundtrip_ok":      bool(roundtrip_ok),
}

metrics = {
    "size":   size,
    "n_rows": int(N_ROWS),
    "common": common,
    "raw": {
        "engine":            "spark",
        "format":            FORMAT,
        "raw_image_gb":      round(raw_image_bytes / 1e9, 4),
        "write_s":           round(write_s, 3),
        "on_disk_bytes":     int(on_disk_bytes) if on_disk_bytes is not None else None,
        "n_output_files":    n_output_files,
        "backfill_s":        round(backfill_s, 3),
        "backfill_via":      backfill_via,
        "etl_bytes_written": int(etl_bytes_written),
        "etl_files":         int(etl_files),
        "n_partitions":      int(n_parts),
        "roundtrip_ok":      bool(roundtrip_ok),
    },
}
out_path = f"{artifacts_dir}/{ARTIFACT_NAME}"
with open(out_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Wrote {out_path}")
print(json.dumps(metrics, indent=2))

## Done — dataset + artifact ready (pure Spark)

The `{FORMAT}` dataset was generated, written, verified, and backfilled **entirely on Spark**. Its
metrics are in `artifacts/{ARTIFACT_NAME}` with the same `common` keys the other formats use, so
`04_compile_results` stacks all three formats directly.

**Next:** run this notebook for the other formats/sizes (or let `02_run_benchmarks` submit the
grid), then `03_training_benchmark` (Ray Train DDP — the natural home for distributed training)
reads each dataset back, and `04_compile_results` compiles the head-to-head.